# AI-Assisted Alert Analysis - Colab Training

Use this notebook to train the calibrated XGBoost entity-matching model in Google Colab, then export the model artifact back to the local FastAPI project.

The API remains an analyst-assistance system. The trained model predicts pair-level `TRUE_POSITIVE` probability; the application still combines that probability with deterministic rules, data quality, historical evidence, and human-review recommendations.

## 1. Runtime

In Colab, choose **Runtime > Change runtime type**.

- CPU is usually enough for 10k-50k synthetic XGBoost rows.
- GPU helps if you enable real `BAAI/bge-m3` sentence-transformer embedding generation later.
- Keep this first pass CPU-friendly by using the repo's deterministic/hash embedding fallback.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the Project Code

Option A: upload a zip of the `ai-alert-analysis` folder to Colab and unzip it.

Option B: clone your repo if it is in GitHub. Replace `YOUR_REPO_URL` below.

In [ ]:
# Option B: clone from GitHub
# !git clone YOUR_REPO_URL /content/ai-alert-analysis

# Option A fallback: after uploading ai-alert-analysis.zip to /content
# !unzip -q /content/ai-alert-analysis.zip -d /content

%cd /content/ai-alert-analysis

## 3. Install Training Dependencies

In [ ]:
!python -m pip install -q -r requirements-train.txt

## 4. Generate Synthetic Training Data

Start with 10,000 rows. Increase to 50,000 once the flow is working.

In [ ]:
!python -m ml.generate_dataset --rows 10000 --output ml/data/synthetic_pairs.jsonl
!wc -l ml/data/synthetic_pairs.jsonl

## 5. Train and Calibrate XGBoost

This creates:

- `ml/models/entity_matcher.joblib`
- `ml/metrics/entity_matcher_metrics.json`

In [ ]:
!python -m ml.train --dataset ml/data/synthetic_pairs.jsonl --output ml/models/entity_matcher.joblib

## 6. Evaluate

In [ ]:
!python -m ml.evaluate --dataset ml/data/synthetic_pairs.jsonl --model ml/models/entity_matcher.joblib

## 7. Inspect Saved Metrics

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path('ml/metrics/entity_matcher_metrics.json').read_text())
metrics

## 8. Sanity Prediction

In [ ]:
import json
from ml.predict import predict

third_party = {
    'entityId': 'TP001',
    'name': 'Gold Mayning 000',
    'country': 'Russia',
    'city': 'Yakutsk',
    'registrationNumber': '1201400007291',
    'taxId': '1435353596',
}
hit = {
    'hitId': 'HIT001',
    'name': 'Gold Mayning 000',
    'country': 'Russia',
    'city': 'Yakutsk',
    'registrationNumber': '1201400007291',
    'taxId': '1435353596',
    'score': 100,
    'level': 'HIGH',
    'listType': 'US_50PLUS',
    'source': 'KHARON',
}
print(json.dumps(predict(third_party, hit), indent=2))

## 9. Export Artifacts to Google Drive

After this cell, download or copy these two files back into the local project:

- `ml/models/entity_matcher.joblib`
- `ml/metrics/entity_matcher_metrics.json`

In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path('/content/drive/MyDrive/ai-alert-analysis/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2('ml/models/entity_matcher.joblib', artifact_dir / 'entity_matcher.joblib')
shutil.copy2('ml/metrics/entity_matcher_metrics.json', artifact_dir / 'entity_matcher_metrics.json')

print(f'Artifacts exported to {artifact_dir}')

## 10. Local Deployment Step

On your local machine, place the downloaded files here:

```text
ai-alert-analysis/ml/models/entity_matcher.joblib
ai-alert-analysis/ml/metrics/entity_matcher_metrics.json
```

Then start the API locally or with Docker. The model service will load `MODEL_PATH=ml/models/entity_matcher.joblib` automatically.